<a href="https://colab.research.google.com/github/jasonsablan03/Horizon-Mart-Customer-Loyalty-Campaign-Analysis/blob/main/customer_loyalty_stats_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
from scipy import stats
from google.colab import files

files.upload()


Saving version1_shopping_behavior_data_final.csv to version1_shopping_behavior_data_final.csv


{'version1_shopping_behavior_data_final.csv': b'customer_id,age,gender,category,purchase_amount_usd,location,review_rating,subscription_status,discount_applied,promo_code_used,previous_purchases,payment_method,purchase_frequency,age_group,subscriber_indicator,promo_indicator,discount_indicator\n1299,29,Male,Clothing,81,Indiana,3.0,No,Yes,Yes,14,Bank Transfer,Fortnightly,26-35,0,1,1\n2712,30,Female,Accessories,22,Nebraska,4.0,No,No,No,50,Venmo,,26-35,0,0,0\n3544,58,Female,Accessories,83,Colorado,4.1,No,No,No,18,Paypal,Every 3 Months,55+,0,0,0\n2866,28,Female,Clothing,97,California,3.7,No,No,No,27,Debit Card,Fortnightly,26-35,0,0,0\n3505,65,Female,Accessories,41,Hawaii,4.0,No,No,No,50,,Monthly,55+,0,0,0\n3817,29,Female,Outerwear,86,Maryland,4.3,No,No,No,45,Credit Card,Fortnightly,26-35,0,0,0\n97,32,,Accessories,73,Delaware,3.6,Yes,Yes,Yes,43,Venmo,Every 3 Months,26-35,1,1,1\n3492,22,Female,Clothing,28,Maine,4.3,No,No,No,15,Debit Card,Monthly,18-25,0,0,0\n1335,46,Male,,52,West Virginia,4.

In [15]:
df = pd.read_csv("version1_shopping_behavior_data_final.csv")
print(df.shape)
df.head()

(3165, 17)


,customer_id,age,gender,category,purchase_amount_usd,location,review_rating,subscription_status,discount_applied,promo_code_used,previous_purchases,payment_method,purchase_frequency,age_group,subscriber_indicator,promo_indicator,discount_indicator
0,1299,29.0,Male,Clothing,81,Indiana,3.0,No,Yes,Yes,14,Bank Transfer,Fortnightly,26-35,0,1,1
1,2712,30.0,Female,Accessories,22,Nebraska,4.0,No,No,No,50,Venmo,NaN,26-35,0,0,0
2,3544,58.0,Female,Accessories,83,Colorado,4.1,No,No,No,18,Paypal,Every 3 Months,55+,0,0,0
3,2866,28.0,Female,Clothing,97,California,3.7,No,No,No,27,Debit Card,Fortnightly,26-35,0,0,0
4,3505,65.0,Female,Accessories,41,Hawaii,4.0,No,No,No,50,NaN,Monthly,55+,0,0,0


In [16]:

##map frequencies to actual numbers
freq_map = {"Weekly": 52, "Bi-Weekly": 26, "Fortnightly": 26,
            "Monthly": 12, "Quarterly": 4, "Every 3 Months": 4, "Annually": 1}

##calculating frequency per year and annual value for the stats test
df["freq_per_year"] = df["purchase_frequency"].map(freq_map)
df["annual_value"] = df["purchase_amount_usd"] * df["freq_per_year"]

##Looking for and omitting na values from our frequencies column
print("Unmapped frequencies:", df["freq_per_year"].isna().sum())
print(df["purchase_frequency"].value_counts())

Unmapped frequencies: 153
purchase_frequency
Every 3 Months    465
Monthly           439
Quarterly         435
Annually          435
Bi-Weekly         420
Fortnightly       410
Weekly            408
Name: count, dtype: int64


In [17]:
print(df["purchase_frequency"].isna().sum())     # should be 153

##cross checking numbers with SQL results
print(df.groupby("subscriber_indicator")["annual_value"].mean())   # expect ~3,096 and ~1,771

##check if dropping the 153 values disproportionately affects one group (sub vs non-sub)
print(df.groupby("subscriber_indicator")["purchase_frequency"].apply(lambda s: s.isna().mean()))

153
subscriber_indicator
0    1862.770871
1    3228.141089
Name: annual_value, dtype: float64
subscriber_indicator
0    0.050818
1    0.041518
Name: purchase_frequency, dtype: float64


In [47]:
##drop the 153 na values and create new dataframe
d = df.dropna(subset=["annual_value"])

##create values for subscribers and non subscribers
sub = d.loc[d.subscriber_indicator == 1, "annual_value"]
non = d.loc[d.subscriber_indicator == 0, "annual_value"]

##cross checking with SQL values
print(len(sub), len(non))
print(sub.mean(), non.mean(), sub.mean() / non.mean() - 1)

##run mann whitney test
stat, p = stats.mannwhitneyu(sub, non, alternative="two-sided")
print("U =", stat, " p =", p)

808 2204
3228.141089108911 1862.7708711433756 0.7329780807273767
U = 908950.5  p = 0.3807572433788132


In [48]:
import numpy as np
##for randomness when rerunning
rng = np.random.default_rng(12)

bootstrap_gaps = []
for i in range(10000):
    subscriber_sample = rng.choice(sub, len(sub))
    non_subscriber_sample = rng.choice(non, len(non))
    gap = subscriber_sample.mean() / non_subscriber_sample.mean() - 1
    bootstrap_gaps.append(gap)

print("95% CI for gap:", np.percentile(bootstrap_gaps, [2.5, 97.5]))


## see if there's a difference between purchase amount and frequency per year in how it affects annual value

# Purchase amount
subscriber_amounts = d.loc[d["subscriber_indicator"] == 1, "purchase_amount_usd"]
non_subscriber_amounts = d.loc[d["subscriber_indicator"] == 0, "purchase_amount_usd"]

amount_test = stats.mannwhitneyu(subscriber_amounts, non_subscriber_amounts)
print("Purchase amount, subscribers:", subscriber_amounts.mean())
print("Purchase amount, non-subscribers:", non_subscriber_amounts.mean())
print("p-value:", amount_test.pvalue)

# Purchase frequency per year
subscriber_freq = d.loc[d["subscriber_indicator"] == 1, "freq_per_year"]
non_subscriber_freq = d.loc[d["subscriber_indicator"] == 0, "freq_per_year"]

freq_test = stats.mannwhitneyu(subscriber_freq, non_subscriber_freq)
print("Frequency, subscribers:", subscriber_freq.mean())
print("Frequency, non-subscribers:", non_subscriber_freq.mean())
print("p-value:", freq_test.pvalue)

95% CI for gap: [-0.18228398  2.37445812]
Purchase amount, subscribers: 126.10272277227723
Purchase amount, non-subscribers: 93.02132486388385
p-value: 0.683003308071003
Frequency, subscribers: 17.986386138613863
Frequency, non-subscribers: 17.044464609800364
p-value: 0.3016101800680825


In [41]:

## look at distribution to see if any outliers are affecting means
print(d["purchase_amount_usd"].describe())
print(d.sort_values("annual_value", ascending=False)
        [["customer_id","subscriber_indicator","purchase_amount_usd","purchase_frequency","annual_value"]].head(15))
print(d.groupby("subscriber_indicator")["annual_value"].median())

count    3012.000000
mean      101.895750
std       585.086173
min         5.000000
25%        37.000000
50%        59.000000
75%        81.000000
max      9999.000000
Name: purchase_amount_usd, dtype: float64
      customer_id  subscriber_indicator  purchase_amount_usd  \
701          3117                     0                 9999   
296           813                     1                 9999   
745          2271                     0                 9999   
293           415                     1                 5000   
1950         1522                     0                 5000   
317            57                     1                 5000   
1978          108                     1                 9999   
2891         2102                     0                 9999   
1988         1041                     1                 9999   
2176          130                     1                 5000   
751          1331                     0                 5000   
2114         3497     

In [32]:
##second test without outliers

d2 = d[d["purchase_amount_usd"] < 1000]
print(len(d), len(d2))

e = d2.loc[d2.subscriber_indicator == 1, "annual_value"]
f = d2.loc[d2.subscriber_indicator == 0, "annual_value"]
print(e.mean(), f.mean(), e.mean() / f.mean() - 1)
print(stats.mannwhitneyu(e, f).pvalue)

##result is still not statistically significant

3012 2994
1048.0525 1003.049225159526 0.04486646688084184
0.5112864890592839


In [45]:
##second CI without outliers
import numpy as np
rng = np.random.default_rng(12)

subscribers = d2.loc[d2["subscriber_indicator"] == 1, "annual_value"]
non_subscribers = d2.loc[d2["subscriber_indicator"] == 0, "annual_value"]

# Bootstrap the gap in average annual value
rng = np.random.default_rng(12)

subscriber_gaps = []
for i in range(10000):
    subscriber_sample = rng.choice(subscribers, len(subscribers))
    non_subscriber_sample = rng.choice(non_subscribers, len(non_subscribers))
    gap = subscriber_sample.mean() / non_subscriber_sample.mean() - 1
    subscriber_gaps.append(gap)

print("95% interval for gap:", np.percentile(subscriber_gaps, [2.5, 97.5]))


# Order value: subscribers vs non-subscribers
sub_orders = d2.loc[d2["subscriber_indicator"] == 1, "purchase_amount_usd"]
non_sub_orders = d2.loc[d2["subscriber_indicator"] == 0, "purchase_amount_usd"]

order_test = stats.mannwhitneyu(sub_orders, non_sub_orders)
print("Order value, subscribers:", sub_orders.mean())
print("Order value, non-subscribers:", non_sub_orders.mean())
print("p-value:", order_test.pvalue)


# Purchase frequency: subscribers vs non-subscribers
sub_freq = d2.loc[d2["subscriber_indicator"] == 1, "freq_per_year"]
non_sub_freq = d2.loc[d2["subscriber_indicator"] == 0, "freq_per_year"]

freq_test = stats.mannwhitneyu(sub_freq, non_sub_freq)
print("Frequency, subscribers:", sub_freq.mean())
print("Frequency, non-subscribers:", non_sub_freq.mean())
print("p-value:", freq_test.pvalue)

95% interval for gap: [-0.04718007  0.14351389]
Order value, subscribers: 58.6175
Order value, non-subscribers: 59.26344576116682
p-value: 0.5218201239702072
Frequency, subscribers: 17.85375
Frequency, non-subscribers: 17.013673655423883
p-value: 0.38300812772659876


In [44]:
##tests for promo and discount vars
discount_users = d2.loc[d2["discount_indicator"] == 1, "annual_value"]
no_discount = d2.loc[d2["discount_indicator"] == 0, "annual_value"]

discount_gaps = []
for i in range(10000):
    discount_sample = rng.choice(discount_users, len(discount_users))
    no_discount_sample = rng.choice(no_discount, len(no_discount))
    gap = discount_sample.mean() / no_discount_sample.mean() - 1
    discount_gaps.append(gap)

discount_test = stats.mannwhitneyu(discount_users, no_discount)

print("Discount users average:", discount_users.mean())
print("Non-users average:", no_discount.mean())
print("Gap:", discount_users.mean() / no_discount.mean() - 1)
print("p-value:", discount_test.pvalue)
print("95% interval:", np.percentile(discount_gaps, [2.5, 97.5]))


# ---------- Promo users vs non-users ----------
promo_users = d2.loc[d2["promo_indicator"] == 1, "annual_value"]
no_promo = d2.loc[d2["promo_indicator"] == 0, "annual_value"]

promo_gaps = []
for i in range(10000):
    promo_sample = rng.choice(promo_users, len(promo_users))
    no_promo_sample = rng.choice(no_promo, len(no_promo))
    gap = promo_sample.mean() / no_promo_sample.mean() - 1
    promo_gaps.append(gap)

promo_test = stats.mannwhitneyu(promo_users, no_promo)

print("Promo users average:", promo_users.mean())
print("Non-users average:", no_promo.mean())
print("Gap:", promo_users.mean() / no_promo.mean() - 1)
print("p-value:", promo_test.pvalue)
print("95% interval:", np.percentile(promo_gaps, [2.5, 97.5]))

Discount users average: 1013.9750195160032
Non-users average: 1015.8960887332166
Gap: -0.001891009561429624
p-value: 0.846411051099329
95% interval: [-0.07927625  0.08450711]
Promo users average: 1008.9714052287582
Non-users average: 1019.2943502824859
Gap: -0.010127540735281104
p-value: 0.7575246584423145
95% interval: [-0.08675932  0.07392572]


In [29]:
df_clean = df[df["purchase_amount_usd"] < 1000].copy()
print(len(df_clean))

3147


In [42]:
rng = np.random.default_rng(12)

male = df_clean[df_clean["gender"] == "Male"]
female = df_clean[df_clean["gender"] == "Female"]

male_orders = male["purchase_amount_usd"]
female_orders = female["purchase_amount_usd"]

order_gaps = []
for i in range(10000):
    male_sample = rng.choice(male_orders, len(male_orders))
    female_sample = rng.choice(female_orders, len(female_orders))
    gap = male_sample.mean() / female_sample.mean() - 1
    order_gaps.append(gap)

print("Male average:", male_orders.mean())
print("Female average:", female_orders.mean())
print("p-value:", stats.mannwhitneyu(male_orders, female_orders).pvalue)
print("95% interval:", np.percentile(order_gaps, [2.5, 97.5]))


Male average: 58.570239843367595
Female average: 59.63096500530223
p-value: 0.2639602802434887
95% interval: [-0.04758038  0.0137204 ]


In [43]:
print(df.groupby("age_group")["age"].agg(["min", "max", "count"]))

            min   max  count
age_group                   
18-25      18.0  25.0    445
26-35      26.0  35.0    551
36-45      36.0  45.0    574
46-54      46.0  54.0    528
55+        55.0  70.0    879
Unknown     NaN   NaN      0


In [40]:
print(df_clean.groupby("age_group")[["purchase_amount_usd", "annual_value"]].mean())
print(df_clean["age_group"].value_counts())
print(df_clean["gender"].value_counts())

order_values = []
annual_values = []

for age_group, data in df_clean.groupby("age_group"):
    if age_group == "Unknown":
        continue
    order_values.append(data["purchase_amount_usd"])
    annual_values.append(data["annual_value"].dropna())

# Kruskal-Wallis tests
order_test = stats.kruskal(*order_values)
annual_test = stats.kruskal(*annual_values)

print("Order value p =", order_test.pvalue)
print("Annual value p =", annual_test.pvalue)

           purchase_amount_usd  annual_value
age_group                                   
18-25                58.770270   1012.785714
26-35                59.679487   1028.306513
36-45                57.766667   1022.542125
46-54                60.051331   1029.234000
55+                  59.490868   1012.956835
Unknown              57.994595    925.901163
age_group
55+        876
36-45      570
26-35      546
46-54      526
18-25      444
Unknown    185
Name: count, dtype: int64
gender
Male      2043
Female     943
Name: count, dtype: int64
Order value p = 0.5187802020643535
Annual value p = 0.3721934140323671
